In [40]:
# ─────────────────────────────────────────────────────────────────────────────
# Imports and helper functions
# ─────────────────────────────────────────────────────────────────────────────
import re
from pathlib import Path

import pandas as pd
import requests

pd.set_option('display.max_columns', None)

def clockToSeconds(clockValue):
    """Convert 'PT12M30.00S' NBA clock string to elapsed seconds in the period."""
    if pd.isna(clockValue) or str(clockValue).strip() == '':
        return pd.NA
    clockText = str(clockValue).replace('PT', '').replace('S', '')
    minutesText, secondsText = clockText.split('M', 1)
    return int(minutesText) * 60 + float(secondsText)


def secondsLeftInGame(periodValue, clockValue, periodTypeValue='REGULAR'):
    """Total seconds remaining in the game at the moment of an action.

    Regulation: 4 quarters × 720 s each.
    Overtime:   5-min periods (300 s); period 5 = OT1, period 6 = OT2, …
    """
    clockSeconds = clockToSeconds(clockValue)
    if str(periodTypeValue).upper() == 'OVERTIME' or periodValue > 4:
        overtimeNumber = max(periodValue - 5, 0)
        return overtimeNumber * 300 + clockSeconds
    return max(4 - periodValue, 0) * 720 + clockSeconds


def loadAllGameMeta(gameIds):
    """Return a DataFrame of home/away metadata for every gameId in *gameIds*.

    Reads local gamelog parquets and returns one row per game with:
      season, gameDate, matchup, homeTeamId, homeAbbreviation,
      awayTeamId, awayAbbreviation, homeWin
    """
    candidateDirs = [Path('nba_gamelog'), Path('data/nba_gamelog')]
    gameIdSet = {str(g) for g in gameIds}

    for candidateDir in candidateDirs:
        paths = sorted(candidateDir.glob('gamelog_*.parquet'), reverse=True)
        if not paths:
            continue

        gamelogDf = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
        gamelogDf.columns = gamelogDf.columns.str.lower().str.replace(
            r'_(\w)', lambda m: m.group(1).upper(), regex=True
        )
        gamelogDf['gameId'] = gamelogDf['gameId'].astype(str)
        gamelogDf = gamelogDf.loc[gamelogDf['gameId'].isin(gameIdSet)]

        if gamelogDf.empty:
            continue

        # Home rows: 'BOS vs. PHI'  |  Away rows: 'PHI @ BOS'
        homeRows = (
            gamelogDf
            .loc[gamelogDf['matchup'].str.contains(' vs. ', na=False),
                 ['gameId', 'season', 'gameDate', 'matchup', 'teamId', 'teamAbbreviation', 'wl']]
            .copy()
            .rename(columns={'teamId': 'homeTeamId', 'teamAbbreviation': 'homeAbbreviation'})
        )
        awayRows = (
            gamelogDf
            .loc[gamelogDf['matchup'].str.contains(' @ ', na=False),
                 ['gameId', 'teamId', 'teamAbbreviation']]
            .copy()
            .rename(columns={'teamId': 'awayTeamId', 'teamAbbreviation': 'awayAbbreviation'})
        )

        metaDf = homeRows.merge(awayRows, on='gameId', how='left')
        metaDf['gameDate'] = pd.to_datetime(metaDf['gameDate']).dt.date
        metaDf['homeWin']  = metaDf['wl'].map({'W': 1, 'L': 0})
        return metaDf.drop(columns=['wl'])

    raise FileNotFoundError(f'No local gamelog found in {[str(d) for d in candidateDirs]}')

In [41]:
# ─────────────────────────────────────────────────────────────────────────────
# Load all 2024-25 GSW play-by-play from the Live API parquet files
# Scraped via scrape_live_gsw_24_25.py using nba_api.live PlayByPlay endpoint.
# gameId is not embedded in the action data, so it is read from the filename.
# ─────────────────────────────────────────────────────────────────────────────
liveDir = Path('live/2024-25/Golden State Warriors')
parquetFiles = sorted(liveDir.glob('*.parquet'))
print(f'Found {len(parquetFiles)} parquet files')

parts = []
for p in parquetFiles:
    gameDf = pd.read_parquet(p)
    gameDf['gameId'] = p.stem   # filename stem = NBA game ID
    # Sort by actionNumber so events are in canonical chronological order.
    # The live API occasionally delivers rows out of order around period
    # transitions (e.g. period-start before end-of-period free throws).
    if 'actionNumber' in gameDf.columns:
        gameDf = gameDf.sort_values('actionNumber').reset_index(drop=True)
    parts.append(gameDf)

gsw_pbp_24_25 = pd.concat(parts, ignore_index=True)
print(f'Total rows : {len(gsw_pbp_24_25):,}')
print(f'Games      : {gsw_pbp_24_25["gameId"].nunique()}')
gsw_pbp_24_25.head(3)

Found 82 parquet files
Total rows : 47,225
Games      : 82


,actionNumber,clock,timeActual,period,periodType,actionType,subType,qualifiers,personId,x,y,possession,scoreHome,scoreAway,edited,orderNumber,isTargetScoreLastPeriod,xLegacy,yLegacy,isFieldGoal,side,description,personIdsFilter,teamId,teamTricode,descriptor,jumpBallRecoveredName,jumpBallRecoverdPersonId,playerName,playerNameI,jumpBallWonPlayerName,jumpBallWonPersonId,jumpBallLostPlayerName,jumpBallLostPersonId,area,areaDetail,officialId,foulPersonalTotal,foulTechnicalTotal,foulDrawnPlayerName,foulDrawnPersonId,shotResult,pointsTotal,shotDistance,assistPlayerNameInitial,assistPersonId,assistTotal,turnoverTotal,stealPlayerName,stealPersonId,shotActionNumber,reboundTotal,reboundDefensiveTotal,reboundOffensiveTotal,blockPlayerName,blockPersonId,gameId
0,2,PT12M00.00S,2024-11-13T03:16:14.4Z,1,REGULAR,period,start,[],0,NaN,NaN,0,0,0,2024-11-13T03:16:14Z,20000,False,NaN,NaN,0,None,Period Start,[],NaN,None,None,None,NaN,None,None,None,NaN,None,NaN,None,None,NaN,NaN,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN,0022400007
1,4,PT11M57.00S,2024-11-13T03:16:16.6Z,1,REGULAR,jumpball,recovered,[],1629029,NaN,NaN,1610612742,0,0,2024-11-13T03:16:16Z,40000,False,NaN,NaN,0,None,Jump Ball D. Gafford vs. T. Jackson-Davis: Tip...,"[1629029, 1629655, 1631218]",1.610613e+09,DAL,startperiod,L. Dončić,1629029.0,Dončić,L. Dončić,Gafford,1629655.0,Jackson-Davis,1631218.0,None,None,NaN,NaN,NaN,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN,0022400007
2,7,PT11M45.00S,2024-11-13T03:16:32.9Z,1,REGULAR,foul,personal,[2freethrow],201939,NaN,NaN,1610612742,0,0,2024-11-13T03:16:41Z,70000,False,NaN,NaN,0,None,S. Curry shooting personal FOUL (1 PF) (Thomps...,"[201939, 202691]",1.610613e+09,GSW,shooting,None,NaN,Curry,S. Curry,None,NaN,None,NaN,Mid-Range,8-16 Center,1627964.0,1.0,0.0,Thompson,202691.0,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN,0022400007


In [42]:
# ─────────────────────────────────────────────────────────────────────────────
# Merge game-level metadata and compute all derived columns
# ─────────────────────────────────────────────────────────────────────────────

# --- Game meta: season, date, matchup, home/away team IDs, win result ----------
metaDf = loadAllGameMeta(gsw_pbp_24_25['gameId'].unique())
gsw_pbp_24_25 = gsw_pbp_24_25.merge(metaDf, on='gameId', how='left')

# --- Scores (arrive as strings from the live API; ffill within each game) ------
# Empty/NaN entries between scoring plays are forward-filled with the last score.
for scoreCol in ['scoreHome', 'scoreAway']:
    if scoreCol in gsw_pbp_24_25.columns:
        gsw_pbp_24_25[scoreCol] = (
            gsw_pbp_24_25
            .groupby('gameId')[scoreCol]
            .transform(lambda s: pd.to_numeric(s, errors='coerce').ffill().fillna(0))
        )

gsw_pbp_24_25['pointsTotal'] = gsw_pbp_24_25['scoreHome'] + gsw_pbp_24_25['scoreAway']
gsw_pbp_24_25['scoreDif']    = gsw_pbp_24_25['scoreHome'] - gsw_pbp_24_25['scoreAway']

# quarter: string label for each period
# Regular periods → 'Q1'…'Q4'; any overtime period → 'OT'
gsw_pbp_24_25['quarter'] = gsw_pbp_24_25['period'].apply(
    lambda p: f'Q{int(p)}' if pd.notna(p) and int(p) <= 4 else 'OT'
)

# --- secondsLeft: total seconds remaining in the game at each action -----------
gsw_pbp_24_25['secondsLeft'] = gsw_pbp_24_25.apply(
    lambda row: secondsLeftInGame(
        row['period'],
        row['clock'],
        row.get('periodType', 'REGULAR')
    ),
    axis=1
)

# --- actionTeamSide: which side (home/away) committed this action --------------
if 'teamId' in gsw_pbp_24_25.columns:
    gsw_pbp_24_25['teamId']     = pd.to_numeric(gsw_pbp_24_25['teamId'],     errors='coerce').astype('Int64')
    gsw_pbp_24_25['homeTeamId'] = pd.to_numeric(gsw_pbp_24_25['homeTeamId'], errors='coerce').astype('Int64')
    gsw_pbp_24_25['awayTeamId'] = pd.to_numeric(gsw_pbp_24_25['awayTeamId'], errors='coerce').astype('Int64')

    gsw_pbp_24_25['isHomeAction']   = gsw_pbp_24_25['teamId'].eq(gsw_pbp_24_25['homeTeamId'])
    gsw_pbp_24_25.loc[gsw_pbp_24_25['teamId'].isna(), 'isHomeAction'] = pd.NA

    gsw_pbp_24_25['actionTeamSide'] = pd.Series(pd.NA, index=gsw_pbp_24_25.index, dtype='object')
    gsw_pbp_24_25.loc[gsw_pbp_24_25['teamId'].eq(gsw_pbp_24_25['homeTeamId']), 'actionTeamSide'] = 'home'
    gsw_pbp_24_25.loc[gsw_pbp_24_25['teamId'].eq(gsw_pbp_24_25['awayTeamId']), 'actionTeamSide'] = 'away'

# --- possessionTeamSide: which side currently has the ball --------------------
if 'possession' in gsw_pbp_24_25.columns:
    gsw_pbp_24_25['possession'] = pd.to_numeric(gsw_pbp_24_25['possession'], errors='coerce').astype('Int64')

    gsw_pbp_24_25['isHomePossession']   = gsw_pbp_24_25['possession'].eq(gsw_pbp_24_25['homeTeamId'])
    gsw_pbp_24_25.loc[gsw_pbp_24_25['possession'].isna(), 'isHomePossession'] = pd.NA

    gsw_pbp_24_25['possessionTeamSide'] = pd.Series(pd.NA, index=gsw_pbp_24_25.index, dtype='object')
    gsw_pbp_24_25.loc[gsw_pbp_24_25['possession'].eq(gsw_pbp_24_25['homeTeamId']), 'possessionTeamSide'] = 'home'
    gsw_pbp_24_25.loc[gsw_pbp_24_25['possession'].eq(gsw_pbp_24_25['awayTeamId']), 'possessionTeamSide'] = 'away'

# --- Reorder: game identifiers first, then action details ---------------------
preferredColumns = [
    'season', 'gameId', 'gameDate', 'matchup',
    'homeTeamId', 'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin',
    'description', 'quarter', 'secondsLeft',
    'scoreHome', 'scoreAway', 'scoreDif', 'pointsTotal',
    'actionTeamSide', 'isHomeAction',
    'possession', 'possessionTeamSide', 'isHomePossession',
    'actionType', 'subType', 'personId', 'playerName',
    'shotResult', 'foulPersonalTotal', 'foulTechnicalTotal',
]
existingCols  = [c for c in preferredColumns if c in gsw_pbp_24_25.columns]
remainingCols = [c for c in gsw_pbp_24_25.columns if c not in existingCols]
gsw_pbp_24_25 = gsw_pbp_24_25[existingCols + remainingCols]

gsw_pbp_24_25.head(3)

,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,quarter,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,isHomeAction,possession,possessionTeamSide,isHomePossession,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,actionNumber,clock,timeActual,period,periodType,qualifiers,x,y,edited,orderNumber,isTargetScoreLastPeriod,xLegacy,yLegacy,isFieldGoal,side,personIdsFilter,teamId,teamTricode,descriptor,jumpBallRecoveredName,jumpBallRecoverdPersonId,playerNameI,jumpBallWonPlayerName,jumpBallWonPersonId,jumpBallLostPlayerName,jumpBallLostPersonId,area,areaDetail,officialId,foulDrawnPlayerName,foulDrawnPersonId,shotDistance,assistPlayerNameInitial,assistPersonId,assistTotal,turnoverTotal,stealPlayerName,stealPersonId,shotActionNumber,reboundTotal,reboundDefensiveTotal,reboundOffensiveTotal,blockPlayerName,blockPersonId
0,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,Period Start,Q1,2880.0,0,0,0,0,<NA>,<NA>,0,<NA>,False,period,start,0,None,None,NaN,NaN,2,PT12M00.00S,2024-11-13T03:16:14.4Z,1,REGULAR,[],NaN,NaN,2024-11-13T03:16:14Z,20000,False,NaN,NaN,0,None,[],<NA>,None,None,None,NaN,None,None,NaN,None,NaN,None,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN
1,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,Jump Ball D. Gafford vs. T. Jackson-Davis: Tip...,Q1,2877.0,0,0,0,0,away,False,1610612742,away,False,jumpball,recovered,1629029,Dončić,None,NaN,NaN,4,PT11M57.00S,2024-11-13T03:16:16.6Z,1,REGULAR,[],NaN,NaN,2024-11-13T03:16:16Z,40000,False,NaN,NaN,0,None,"[1629029, 1629655, 1631218]",1610612742,DAL,startperiod,L. Dončić,1629029.0,L. Dončić,Gafford,1629655.0,Jackson-Davis,1631218.0,None,None,NaN,None,NaN,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN
2,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,S. Curry shooting personal FOUL (1 PF) (Thomps...,Q1,2865.0,0,0,0,0,home,True,1610612742,away,False,foul,personal,201939,Curry,None,1.0,0.0,7,PT11M45.00S,2024-11-13T03:16:32.9Z,1,REGULAR,[2freethrow],NaN,NaN,2024-11-13T03:16:41Z,70000,False,NaN,NaN,0,None,"[201939, 202691]",1610612744,GSW,shooting,None,NaN,S. Curry,None,NaN,None,NaN,Mid-Range,8-16 Center,1627964.0,Thompson,202691.0,NaN,None,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,NaN


In [43]:
# ─────────────────────────────────────────────────────────────────────────────
# Drop raw/granular columns not needed downstream
# ─────────────────────────────────────────────────────────────────────────────
colsToDrop = [
    # superseded by derived columns
    'isHomePossession', 'isHomeAction', 'teamId', 'teamTricode', 'period',
    'actionNumber', 'clock', 'timeActual', 'periodType',
    # positional / shot-chart
    'x', 'y', 'xLegacy', 'yLegacy', 'shotDistance', 'isFieldGoal', 'area', 'areaDetail',
    # misc flags & filters
    'edited', 'orderNumber', 'qualifiers', 'side', 'personIdsFilter', 'descriptor',
    'isTargetScoreLastPeriod',
    # jump ball detail
    'jumpBallRecoveredName', 'jumpBallRecoverdPersonId', 'playerNameI',
    'jumpBallWonPlayerName', 'jumpBallWonPersonId',
    'jumpBallLostPlayerName', 'jumpBallLostPersonId',
    # shot / rebound / assist detail
    'shotActionNumber', 'reboundTotal', 'reboundDefensiveTotal', 'reboundOffensiveTotal',
    'blockPlayerName', 'blockPersonId',
    'assistPlayerNameInitial', 'assistPersonId', 'assistTotal',
    # foul / turnover / steal detail
    'foulDrawnPlayerName', 'foulDrawnPersonId',
    'stealPlayerName', 'stealPersonId',
    'officialId', 'turnoverTotal',
]
colsToDrop = [c for c in colsToDrop if c in gsw_pbp_24_25.columns]
gsw_pbp_24_25 = gsw_pbp_24_25.drop(columns=colsToDrop)

print(gsw_pbp_24_25.columns.tolist())

['season', 'gameId', 'gameDate', 'matchup', 'homeTeamId', 'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin', 'description', 'quarter', 'secondsLeft', 'scoreHome', 'scoreAway', 'scoreDif', 'pointsTotal', 'actionTeamSide', 'possession', 'possessionTeamSide', 'actionType', 'subType', 'personId', 'playerName', 'shotResult', 'foulPersonalTotal', 'foulTechnicalTotal']


In [44]:
# ─────────────────────────────────────────────────────────────────────────────
# Merge Vegas betting lines from Rotowire
# Matched on gameDate × homeAbbreviation × awayAbbreviation.
# If the request fails the rest of the pipeline continues without the 'line' column.
# ─────────────────────────────────────────────────────────────────────────────
rotowireUrl = 'https://www.rotowire.com/betting/nba/tables/games-archive.php'
try:
    resp = requests.get(rotowireUrl, timeout=30)
    resp.raise_for_status()
    rotowireDf = (
        pd.DataFrame(resp.json())
        [['game_date', 'home_team_abbrev', 'visit_team_abbrev', 'line']]
        .rename(columns={
            'game_date':         'gameDate',
            'home_team_abbrev':  'homeAbbreviation',
            'visit_team_abbrev': 'awayAbbreviation',
        })
    )
    rotowireDf['gameDate'] = pd.to_datetime(rotowireDf['gameDate']).dt.date
    rotowireDf = rotowireDf.drop_duplicates(subset=['gameDate', 'homeAbbreviation', 'awayAbbreviation'])
    gsw_pbp_24_25 = gsw_pbp_24_25.merge(
        rotowireDf, on=['gameDate', 'homeAbbreviation', 'awayAbbreviation'], how='left'
    )
    print('Rotowire merge successful')
except Exception as exc:
    print(f'Rotowire merge skipped: {exc}')

gsw_pbp_24_25.head(3)

Rotowire merge successful


,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,quarter,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,possessionTeamSide,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,line
0,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,Period Start,Q1,2880.0,0,0,0,0,<NA>,0,<NA>,period,start,0,None,None,NaN,NaN,-2.5
1,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,Jump Ball D. Gafford vs. T. Jackson-Davis: Tip...,Q1,2877.0,0,0,0,0,away,1610612742,away,jumpball,recovered,1629029,Dončić,None,NaN,NaN,-2.5
2,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,S. Curry shooting personal FOUL (1 PF) (Thomps...,Q1,2865.0,0,0,0,0,home,1610612742,away,foul,personal,201939,Curry,None,1.0,0.0,-2.5


In [45]:
# ─────────────────────────────────────────────────────────────────────────────
# Per-team, per-quarter foul counts and bonus indicators
# ─────────────────────────────────────────────────────────────────────────────

# foul_df: one row per personal foul (technicals excluded — they do not count
# toward the team-foul limit that triggers the bonus)
foul_df = gsw_pbp_24_25.loc[
    (gsw_pbp_24_25['actionType'] == 'foul') & (gsw_pbp_24_25['subType'] != 'technical'),
    ['gameId', 'personId', 'actionTeamSide', 'playerName', 'quarter', 'secondsLeft', 'foulPersonalTotal']
].copy()
foul_df = foul_df.rename(columns={'actionTeamSide': 'playerHomeAway'})

# For each action, count how many personal fouls each side has committed
# within the same game × quarter up to (and including) that moment.
#
# secondsLeft counts DOWN as the game progresses, so a foul with
# secondsLeft_foul >= secondsLeft_action happened at or before the action.
home_foul_events = foul_df[foul_df['playerHomeAway'] == 'home'][['gameId', 'quarter', 'secondsLeft']].copy()
away_foul_events = foul_df[foul_df['playerHomeAway'] == 'away'][['gameId', 'quarter', 'secondsLeft']].copy()

df_idx = gsw_pbp_24_25[['gameId', 'quarter', 'secondsLeft']].reset_index()

home_merged = df_idx.merge(home_foul_events, on=['gameId', 'quarter'], suffixes=('', '_foul'))
home_merged = home_merged[home_merged['secondsLeft_foul'] >= home_merged['secondsLeft']]
gsw_pbp_24_25['homeFouls'] = home_merged.groupby('index').size().reindex(gsw_pbp_24_25.index, fill_value=0)

away_merged = df_idx.merge(away_foul_events, on=['gameId', 'quarter'], suffixes=('', '_foul'))
away_merged = away_merged[away_merged['secondsLeft_foul'] >= away_merged['secondsLeft']]
gsw_pbp_24_25['awayFouls'] = away_merged.groupby('index').size().reindex(gsw_pbp_24_25.index, fill_value=0)

# homeBonus: away team has 5+ quarter fouls → home team is in the bonus
# awayBonus: home team has 5+ quarter fouls → away team is in the bonus
gsw_pbp_24_25['homeBonus'] = (gsw_pbp_24_25['awayFouls'] >= 5).astype(int)
gsw_pbp_24_25['awayBonus'] = (gsw_pbp_24_25['homeFouls'] >= 5).astype(int)

gsw_pbp_24_25[['gameId', 'quarter', 'secondsLeft', 'homeFouls', 'awayFouls', 'homeBonus', 'awayBonus']].head(20)

,gameId,quarter,secondsLeft,homeFouls,awayFouls,homeBonus,awayBonus
0,0022400007,Q1,2880.0,0,0,0,0
1,0022400007,Q1,2877.0,0,0,0,0
2,0022400007,Q1,2865.0,1,0,0,0
3,0022400007,Q1,2865.0,1,0,0,0
4,0022400007,Q1,2865.0,1,0,0,0
5,0022400007,Q1,2848.0,1,0,0,0
6,0022400007,Q1,2830.0,1,0,0,0
7,0022400007,Q1,2830.0,1,0,0,0
8,0022400007,Q1,2824.0,1,0,0,0
9,0022400007,Q1,2813.0,1,0,0,0


In [46]:
# ─────────────────────────────────────────────────────────────────────────────
# Per-game free throw tracking  (homeFreeThrows / awayFreeThrows)
# ─────────────────────────────────────────────────────────────────────────────
# Each column holds the number of free throws the team still has to shoot
# AFTER the action on that row.
#
# State-machine rules (applied row-by-row, resetting between games):
#
#   period start         → error if either counter is non-zero (data integrity check)
#   technical foul       → opposing team gets 1 FT
#   other foul w/ (M FT) → fouled team gets M FTs
#   freethrow 'X of M'   → shooting team's counter set to M - X (remaining)



ft_awarded_pattern = re.compile(r'\((\w[\w\s\.]*\s)?(\d+)\s+FT\)')
ft_attempt_pattern  = re.compile(r'(\d+)\s+of\s+(\d+)')


def computeFreeThrowsForGame(game_df):
    """Run the FT state machine over one game's rows in chronological order."""
    home_ft, away_ft = 0, 0
    home_ft_list, away_ft_list = [], []

    for _, row in game_df.iterrows():
        action = str(row['actionType']) if pd.notna(row['actionType']) else ''
        sub    = str(row['subType'])    if pd.notna(row['subType'])    else ''
        desc   = str(row['description']) if pd.notna(row['description']) else ''
        side   = row['actionTeamSide']   # 'home', 'away', or NA

        if action == 'period' and sub == 'start':
            # FTs straddling a period boundary are legitimate in NBA data:
            # end-of-period fouls and tipoff technicals both produce FT events
            # whose actionNumbers place them across the period-start event.
            # Log a warning but let the state machine continue; the FT events
            # that follow will decrement the counters normally.
            if home_ft != 0 or away_ft != 0:
                print(
                    f"[game {row['gameId']}] WARNING: period started with "
                    f"homeFreeThrows={home_ft}, awayFreeThrows={away_ft} — "
                    f"FTs straddle a period boundary (technical foul before quarter start)."
                )

        elif action == 'foul' and sub == 'technical':
            # Double technical fouls cancel each other out — no FTs are shot.
            # Single technicals award exactly 1 FT to the opposing team.
            if 'double' not in desc.lower():
                if side == 'home':
                    away_ft = 1
                elif side == 'away':
                    home_ft = 1

        elif action == 'foul':
            # Non-technical: look for (M FT) in description
            ft_match = ft_awarded_pattern.search(desc)
            if ft_match:
                m = int(ft_match.group(2))
                if side == 'home':
                    away_ft = m
                elif side == 'away':
                    home_ft = m

        elif action == 'freethrow':
            # subType 'X of M': after attempt X, M - X remain
            of_match = ft_attempt_pattern.search(sub)
            if of_match:
                x = int(of_match.group(1))
                m = int(of_match.group(2))
                remaining = m - x
                if side == 'home':
                    home_ft = remaining
                elif side == 'away':
                    away_ft = remaining

        home_ft_list.append(home_ft)
        away_ft_list.append(away_ft)

    result = game_df.copy()
    result['homeFreeThrows'] = home_ft_list
    result['awayFreeThrows'] = away_ft_list
    return result


# Apply per-game, then reassemble preserving original row order
game_parts = [
    computeFreeThrowsForGame(game_df)
    for _, game_df in gsw_pbp_24_25.groupby('gameId', sort=False)
]
gsw_pbp_24_25 = pd.concat(game_parts).sort_index()

# Spot-check: foul and freethrow rows from the first game
first_game = gsw_pbp_24_25['gameId'].iloc[0]
spot_check = gsw_pbp_24_25.loc[
    (gsw_pbp_24_25['gameId'] == first_game) &
    gsw_pbp_24_25['actionType'].isin(['foul', 'freethrow'])
].head(20)
spot_check[['description', 'actionType', 'subType', 'actionTeamSide', 'homeFreeThrows', 'awayFreeThrows']]

[game 0022400957] WARNING: period started with homeFreeThrows=1, awayFreeThrows=0 — FTs straddle a period boundary (technical foul before quarter start).


,description,actionType,subType,actionTeamSide,homeFreeThrows,awayFreeThrows
2,S. Curry shooting personal FOUL (1 PF) (Thomps...,foul,personal,home,0,2
3,K. Thompson Free Throw 1 of 2 (1 PTS),freethrow,1 of 2,away,0,1
4,K. Thompson Free Throw 2 of 2 (2 PTS),freethrow,2 of 2,away,0,0
12,A. Wiggins shooting personal FOUL (1 PF) (Mars...,foul,personal,home,0,2
13,N. Marshall Free Throw 1 of 2 (1 PTS),freethrow,1 of 2,away,0,1
14,N. Marshall Free Throw 2 of 2 (2 PTS),freethrow,2 of 2,away,0,0
34,L. Dončić shooting personal FOUL (1 PF) (Jacks...,foul,personal,away,0,0
35,T. Jackson-Davis Free Throw 1 of 2 (1 PTS),freethrow,1 of 2,home,1,0
36,T. Jackson-Davis Free Throw 2 of 2 (2 PTS),freethrow,2 of 2,home,0,0
44,T. Jackson-Davis personal FOUL (1 PF),foul,personal,home,0,0


In [47]:
# ─────────────────────────────────────────────────────────────────────────────
# Final summary
# ─────────────────────────────────────────────────────────────────────────────
print(f'Shape  : {gsw_pbp_24_25.shape}')
print(f'Games  : {gsw_pbp_24_25["gameId"].nunique()}')
print(f'Columns: {gsw_pbp_24_25.columns.tolist()}')
gsw_pbp_24_25.head()

Shape  : (47225, 33)
Games  : 82
Columns: ['season', 'gameId', 'gameDate', 'matchup', 'homeTeamId', 'homeAbbreviation', 'awayTeamId', 'awayAbbreviation', 'homeWin', 'description', 'quarter', 'secondsLeft', 'scoreHome', 'scoreAway', 'scoreDif', 'pointsTotal', 'actionTeamSide', 'possession', 'possessionTeamSide', 'actionType', 'subType', 'personId', 'playerName', 'shotResult', 'foulPersonalTotal', 'foulTechnicalTotal', 'line', 'homeFouls', 'awayFouls', 'homeBonus', 'awayBonus', 'homeFreeThrows', 'awayFreeThrows']


,season,gameId,gameDate,matchup,homeTeamId,homeAbbreviation,awayTeamId,awayAbbreviation,homeWin,description,quarter,secondsLeft,scoreHome,scoreAway,scoreDif,pointsTotal,actionTeamSide,possession,possessionTeamSide,actionType,subType,personId,playerName,shotResult,foulPersonalTotal,foulTechnicalTotal,line,homeFouls,awayFouls,homeBonus,awayBonus,homeFreeThrows,awayFreeThrows
0,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,Period Start,Q1,2880.0,0,0,0,0,<NA>,0,<NA>,period,start,0,None,None,NaN,NaN,-2.5,0,0,0,0,0,0
1,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,Jump Ball D. Gafford vs. T. Jackson-Davis: Tip...,Q1,2877.0,0,0,0,0,away,1610612742,away,jumpball,recovered,1629029,Dončić,None,NaN,NaN,-2.5,0,0,0,0,0,0
2,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,S. Curry shooting personal FOUL (1 PF) (Thomps...,Q1,2865.0,0,0,0,0,home,1610612742,away,foul,personal,201939,Curry,None,1.0,0.0,-2.5,1,0,0,0,0,2
3,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,K. Thompson Free Throw 1 of 2 (1 PTS),Q1,2865.0,0,1,-1,1,away,1610612742,away,freethrow,1 of 2,202691,Thompson,Made,NaN,NaN,-2.5,1,0,0,0,0,1
4,2024-25,0022400007,2024-11-12,GSW vs. DAL,1610612744,GSW,1610612742,DAL,1,K. Thompson Free Throw 2 of 2 (2 PTS),Q1,2865.0,0,2,-2,2,away,1610612742,away,freethrow,2 of 2,202691,Thompson,Made,NaN,NaN,-2.5,1,0,0,0,0,0


In [48]:
gsw_pbp_24_25.quarter.drop_duplicates()

0       Q1
150     Q2
292     Q3
454     Q4
6043    OT
Name: quarter, dtype: object

## Column Descriptions

| Column | Description |
|---|---|
| `season` | NBA season identifier (e.g. `"2024-25"`) |
| `gameId` | Unique game identifier from the NBA API |
| `gameDate` | Date the game was played |
| `matchup` | Human-readable matchup string from the home team's perspective (e.g. `"GSW vs. DAL"`) |
| `homeTeamId` | NBA team ID for the home team |
| `homeAbbreviation` | Three-letter abbreviation for the home team (e.g. `"GSW"`) |
| `awayTeamId` | NBA team ID for the away team |
| `awayAbbreviation` | Three-letter abbreviation for the away team (e.g. `"DAL"`) |
| `homeWin` | `1` if the home team won, `0` otherwise |
| `description` | Raw play description string from the NBA Live API |
| `quarter` | Period label: `"Q1"`–`"Q4"` for regulation, `"OT"` for any overtime period |
| `secondsLeft` | Seconds remaining in the current period at the time of the action |
| `scoreHome` | Cumulative home team score at this action (forward-filled from scoring plays) |
| `scoreAway` | Cumulative away team score at this action (forward-filled from scoring plays) |
| `scoreDif` | `scoreHome - scoreAway` — positive means home team is leading |
| `pointsTotal` | Combined score at this action (`scoreHome + scoreAway`) |
| `actionTeamSide` | Side that committed the action: `"home"`, `"away"`, or `NaN` for team-neutral events |
| `possession` | Raw team ID of the team with possession (from NBA Live API; `NaN` when unresolved) |
| `possessionTeamSide` | Which side has possession: `"home"`, `"away"`, or `NaN` when unresolved |
| `actionType` | Broad category of the play (e.g. `"foul"`, `"freethrow"`, `"2pt"`, `"3pt"`, `"rebound"`, `"turnover"`) |
| `subType` | More specific classification of the play (e.g. `"personal"`, `"technical"`, `"1 of 2"`) |
| `personId` | NBA player ID of the primary player involved in the action |
| `playerName` | Last name of the primary player involved in the action |
| `shotResult` | `"Made"` or `"Missed"` for shot actions; `NaN` otherwise |
| `foulPersonalTotal` | Cumulative personal foul count for the fouling player this game (sourced from NBA Live API) |
| `foulTechnicalTotal` | Cumulative technical foul count for the fouling player this game (sourced from NBA Live API) |
| `line` | Rotowire pre-game betting line (home-team point spread; e.g. `-3` means home team favored by 3) |
| `homeFouls` | Personal fouls committed by the **home team** in the current quarter up to and including this action |
| `awayFouls` | Personal fouls committed by the **away team** in the current quarter up to and including this action |
| `homeBonus` | `1` if the home team is in the bonus (`awayFouls >= 5` in the current quarter), `0` otherwise |
| `awayBonus` | `1` if the away team is in the bonus (`homeFouls >= 5` in the current quarter), `0` otherwise |
| `homeFreeThrows` | Free throws the **home team** still has to shoot **after** this action (state-machine running count) |
| `awayFreeThrows` | Free throws the **away team** still has to shoot **after** this action (state-machine running count) |

---

## Concerns / Open Questions

### 1. Bonus threshold: 4 fouls or 5?
- Bonus is currently set whenever there are 5 fouls. However, after the 4th foul it may be better to set the bonus flag because the next foul causes free throws

### 2. Bonus in overtime is not handled separately
- need to fix

### 3. Are technical fouls included in `foulPersonalTotal`?
- Check NBA rules and current implementation

### 4. Flagrant foul rules not explicitly modeled
- Check whether flagrant foul free throws are properly handled

### 5. `possessionTeamSide` is `NaN` for many rows
- possession is home, away, or NA. may need to change this

### 6. Ejections not tracked
- Need to account for player foul count (which is easy given what we are already doing) and other types of ejections. Check if there are any ejections clearly stated in the data